# SafeScan Inference Server
**FastAPI + ngrok tunnel for Flutter app**

### Setup Order
1. Connect GPU first (Runtime → Change runtime type → T4)
2. Run Cell 1 → Restart runtime
3. Run Cell 2 → symlink fix
4. Run Cell 3 → load model
5. Run Cell 4 → test locally
6. Run Cell 5 → start server
7. Copy ngrok URL → paste into Flutter safescan_service.dart

## Cell 1 — Install Dependencies
Run once, then restart runtime.

In [3]:
import sys

# Core ML stack
!{sys.executable} -m pip install -q \
    transformers \
    peft \
    accelerate \
    datasets \
    scipy einops sentencepiece

# bitsandbytes
!{sys.executable} -m pip install -q bitsandbytes==0.46.1

# Server stack
!{sys.executable} -m pip install -q \
    fastapi \
    uvicorn \
    pyngrok \
    nest-asyncio \
    httpx

print('✅ All packages installed.')
print('⚠️  Restart runtime now, then run Cell 2 onwards.')

✅ All packages installed.
⚠️  Restart runtime now, then run Cell 2 onwards.


## Cell 2 — Fix CUDA 13 + Verify GPU

In [1]:
import os, torch

# Symlink bitsandbytes for CUDA 13
src = '/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda129.so'
dst = '/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda130.so'
if os.path.exists(src) and not os.path.exists(dst):
    os.symlink(src, dst)
    print('✅ CUDA 13 symlink created')
else:
    print('✅ Symlink already exists or not needed')

# GPU check
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1024**3
    print(f'✅ GPU:     {props.name}')
    print(f'✅ VRAM:    {vram:.1f} GB')
    print(f'✅ CUDA:    {torch.version.cuda}')
    print(f'✅ PyTorch: {torch.__version__}')
else:
    print('❌ No GPU detected — connect T4 first')

✅ Symlink already exists or not needed
✅ GPU:     Tesla T4
✅ VRAM:    14.6 GB
✅ CUDA:    12.8
✅ PyTorch: 2.11.0+cu128


## Cell 3 — Load Model + Adapters from HuggingFace *

In [5]:
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = 'MuhammadSanan99989/safescan-phi3-mini-intent-gemini'

print("🔄 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=False  # Avoids buggy remote tokenizers
)
tokenizer.padding_side = 'right'

print("🛠️ Pre-configuring built-in Phi-3 parameters...")
config = AutoConfig.from_pretrained(MODEL_PATH)

# Enforce native transformers layout instead of pulling Microsoft's custom repo scripts
if hasattr(config, "auto_map"):
    delattr(config, "auto_map")

# Standardize RoPE formatting mapping for the native architecture
if hasattr(config, "rope_scaling") and config.rope_scaling is not None:
    if "rope_type" not in config.rope_scaling:
        config.rope_scaling["rope_type"] = config.rope_scaling.get("type", "default")

print("🔄 Loading model weights via native transformers API (Native FP16)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    config=config,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=False,  # CRITICAL: Forces Hugging Face to use its own updated internal Phi3 code
    attn_implementation='eager'
)

model.config.use_cache = True
model.eval()

print('✅ Standalone model loaded securely via native architecture!')
print(f'   VRAM footprint: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-1' coro=<Server.serve() done, defined at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:77> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_5896/2805076329.py", line 65, in <cell line: 0>
    loop.run_until_complete(server.serve())
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 133, in _run_once
    handle._run()
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "/usr/lib/python3.12/asyncio/tasks.py", line 396, in __wakeup
    self.__step()
  File "/usr/lib/python3.12/asyncio/tasks.py"

🔄 Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


🛠️ Pre-configuring built-in Phi-3 parameters...
🔄 Loading model weights via native transformers API (Native FP16)...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Standalone model loaded securely via native architecture!
   VRAM footprint: 7.12 GB


## Cell 4 — Define Predict Function + Local Test *

In [5]:
import re
import json
import torch

# Dynamically resolve the absolute cleanest stop token sequence available
STOP_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|end|>")
if STOP_TOKEN_ID is None or STOP_TOKEN_ID == tokenizer.unk_token_id:
    STOP_TOKEN_ID = tokenizer.eos_token_id

def predict(user_text: str) -> dict:
    """Run deterministic intent routing with aggressive parsing fallback layers."""

    # Mirror the exact raw layout the model bonded to during training
    prompt = f'<|user|>\n<|user|>\n{user_text}<|end|>\n<|assistant|><|end|>\n<|assistant|>'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=250,          # Bumped up to give complex parameter trees room to breathe
            do_sample=False,             # Pure greedy decoding
            eos_token_id=STOP_TOKEN_ID,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    generated_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Clean up markdown code blocks if the model tried to get fancy
    decoded = re.sub(r'^```json\s*|```$', '', decoded, flags=re.MULTILINE).strip()

    # Attempt a clean, valid structure recovery
    if '}' in decoded:
        cleaned_json = decoded.split('}')[0] + '}'
    else:
        cleaned_json = decoded

    try:
        return json.loads(cleaned_json)
    except json.JSONDecodeError:
        # 🩹 REGEX SALVAGE LAYER: Extract the intent string directly from the broken payload
        intent_match = re.search(r'"intent":\s*"([^"]+)"', decoded)
        if intent_match:
            return {
                'intent': intent_match.group(1).strip(),
                'module': 'SalvagedPayload',
                'action': 'regex_extraction_fallback',
                'parameters': {},
                'raw_salvage': decoded
            }

        # Absolute structural disaster fallback
        return {
            'intent': 'parse_error',
            'raw_output': decoded
        }

# ── Local Test Verification Matrix ─────────────────────────────────────────────
test_cases = [
    ('scan my wifi network',               'wifi_check'),
    ('is my device secure',                'device_check'),
    ('check SSL certificate for site',    'ssl_check'),
    ('have i been breached',              'breach_lookup'),
    ('what permissions does this app use','permission_audit'),
    ('hello how are you',                 'general_chat'),
    ('generate security report',          'report_action'),
    ('how do i use this app',             'interface_help'),
    ('order me a pizza',                  'out_of_scope'),
]

print('🧪 Running heavy-duty salvage verification matrix...')
print('=' * 65)
correct = 0
for text, expected in test_cases:
    result   = predict(text)
    intent   = result.get('intent', 'parse_error')
    status   = '✅' if intent == expected else '❌'
    correct += 1 if intent == expected else 0
    print(f'{status}  {text:<42} → {intent}')
print('=' * 65)
print(f'🎯 Training Alignment Verification Score: {correct}/{len(test_cases)}')

🧪 Running heavy-duty salvage verification matrix...
✅  scan my wifi network                       → wifi_check
✅  is my device secure                        → device_check
✅  check SSL certificate for site             → ssl_check
✅  have i been breached                       → breach_lookup
✅  what permissions does this app use         → permission_audit
✅  hello how are you                          → general_chat
✅  generate security report                   → report_action
❌  how do i use this app                      → out_of_scope
✅  order me a pizza                           → out_of_scope
🎯 Training Alignment Verification Score: 8/9


## Cell 5 — Start FastAPI Server + ngrok Tunnel
Replace `YOUR_NGROK_TOKEN` with your token from https://ngrok.com

The printed URL goes into Flutter's `safescan_service.dart` as the base URL.

In [ ]:
import asyncio
import nest_asyncio
import uvicorn
import re
import json
import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForCausalLM

# Patch the event loop to allow nested loops inside Jupyter/Colab
nest_asyncio.apply()

# ==========================================================
# BULLETPROOF VARIABLE INITIALIZATION LAYER
# ==========================================================
HUGGINGFACE_REPO = 'MuhammadSanan99989/safescan-phi3-mini-intent-gemini'

# Check if the variables exist in the global scope; if not, load them right here
if 'eval_tokenizer' not in globals() or 'eval_model' not in globals():
    print("🔄 Session memory empty. Automatically loading tokenizer and model weights...")

    eval_tokenizer = AutoTokenizer.from_pretrained(HUGGINGFACE_REPO, trust_remote_code=False)

    # Check if a GPU is available to prevent slow CPU generation bottlenecks
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"📡 Loading model onto device target: {device}")

    eval_model = AutoModelForCausalLM.from_pretrained(
        HUGGINGFACE_REPO,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto"
    )
    eval_model.eval()
    print("✅ Model components loaded successfully into memory.")
else:
    print("✅ Active model weights found in memory scope. Skipping initialization step.")

# Resolve the stop token structure
STOP_TOKEN_ID = eval_tokenizer.convert_tokens_to_ids("<|end|>")
if STOP_TOKEN_ID is None or STOP_TOKEN_ID == eval_tokenizer.unk_token_id:
    STOP_TOKEN_ID = eval_tokenizer.eos_token_id

# ==========================================================
# 🔮 INFERENCE PREDICT FUNCTION
# ==========================================================
def predict(user_text: str) -> dict:
    """Run deterministic intent routing with aggressive parsing fallback layers."""
    prompt = f'<|user|>\n<|user|>\n{user_text}<|end|>\n<|assistant|><|end|>\n<|assistant|>'
    inputs = eval_tokenizer(prompt, return_tensors='pt').to(eval_model.device)

    with torch.no_grad():
        output = eval_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            eos_token_id=STOP_TOKEN_ID,
            pad_token_id=eval_tokenizer.eos_token_id,
            use_cache=True,
        )

    generated_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = eval_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    decoded = re.sub(r'^```json\s*|```$', '', decoded, flags=re.MULTILINE).strip()

    if '}' in decoded:
        cleaned_json = decoded.split('}')[0] + '}'
    else:
        cleaned_json = decoded

    try:
        return json.loads(cleaned_json)
    except json.JSONDecodeError:
        intent_match = re.search(r'"intent":\s*"([^"]+)"', decoded)
        if intent_match:
            return {
                'intent': intent_match.group(1).strip(),
                'module': 'SalvagedPayload',
                'action': 'regex_extraction_fallback',
                'parameters': {},
                'raw_salvage': decoded
            }
        return {
            'intent': 'parse_error',
            'raw_output': decoded
        }

# ==========================================================
# 📡 FASTAPI / UVICORN PIPELINE
# ==========================================================
app = FastAPI(title='SafeScan Inference API Router', description='Production API Pipeline')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

class PredictRequest(BaseModel):
    message: str

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'safescan-phi3-mini-merged'}

@app.post('/predict')
def predict_endpoint(req: PredictRequest):
    if not req.message.strip():
        raise HTTPException(status_code=400, detail='Message string cannot be empty.')

    result = predict(req.message)
    return result

# ── Launch Public Ngrok Tunnel ───────────────────────────────────────────────
NGROK_TOKEN = '3EV1QEBC7sqb4CbkmoMcfsY7rla_6yxALmzXQ1J27NcL97iGg'
ngrok.set_auth_token(NGROK_TOKEN)

# Drop any zombie ngrok connections
ngrok.kill()

public_url = ngrok.connect(8000)
clean_url = str(public_url).split('"')[1] if '"' in str(public_url) else str(public_url).replace('NgrokTunnel: ', '').split(' -> ')[0]

print('=' * 65)
print(f'🚀 SafeScan Production Server is LIVE')
print(f'🌐 Base Tunnel Domain : {clean_url}')
print(f'📡 Flutter App Target : {clean_url}/predict')
print('=' * 65)
print('Streaming real-time request logs below. Press the STOP button to terminate.')
print('=' * 65)

# ── Real-Time Streaming Event Loop ───────────────────────────────────────────
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())

🔄 Session memory empty. Automatically loading tokenizer and model weights...
📡 Loading model onto device target: cuda


/usr/lib/python3.12/contextlib.py:132: RuntimeWarning: coroutine 'Server.serve' was never awaited
  def __enter__(self):
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model components loaded successfully into memory.
🚀 SafeScan Production Server is LIVE
🌐 Base Tunnel Domain : https://voter-cardiac-roundish.ngrok-free.dev
📡 Flutter App Target : https://voter-cardiac-roundish.ngrok-free.dev/predict
Streaming real-time request logs below. Press the STOP button to terminate.


INFO:     Started server process [5896]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     139.135.59.39:0 - "POST /predi

## Cell 6 — Test Server (run in separate tab while Cell 5 is running)

In [9]:
import requests

# Test the Health Endpoint
health_resp = requests.get("https://voter-cardiac-roundish.ngrok-free.dev/health")
print("Health Check:", health_resp.json())

# Test the Predict Endpoint
payload = {"message": "scan my wifi network"}
predict_resp = requests.post("https://voter-cardiac-roundish.ngrok-free.dev/predict", json=payload)
print("Predict Routing:", predict_resp.json())

JSONDecodeError: Expecting value: line 1 column 1 (char 0)